In [18]:
import json
import pandas as pd
from IPython.display import display, HTML
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
import os

def load_result():
    possible_paths = [
        "result.json",
        "outputs/result.json"
    ]
    for p in possible_paths:
        if os.path.exists(p):
            with open(p, encoding='utf-8') as f:
                return json.load(f)
    raise Exception("not finding result.json")

data = load_result()
summary = data.get("summary", {})
parsed_trades = data.get("parsed_trades", [])
compliance_results = data.get("compliance_results", [])
novel_trade_ids = summary.get("novel_instrument_trade_ids", [])

In [19]:
# ======================
# Chart 1：Portfolio compliance heatmap:
print("1. Portfolio compliance heatmap")

compliance_map = {}
for c in compliance_results:
    tid = c["trade_id"]
    regime = c["regime"]
    status = c["status"]
    if tid not in compliance_map:
        compliance_map[tid] = {}
    compliance_map[tid][regime] = status

trade_ids = [t["trade_id"] for t in parsed_trades]
heatmap_data = []
annotations = []

for tid in trade_ids:
    cftc = compliance_map.get(tid, {}).get("CFTC", "NONCOMPLIANT")
    emir = compliance_map.get(tid, {}).get("EMIR", "NONCOMPLIANT")
    val_cftc = 2 if cftc == "PASS" else 0
    val_emir = 2 if emir == "PASS" else 0
    heatmap_data.append([val_cftc, val_emir])
    annotations.append([cftc, emir])

fig = ff.create_annotated_heatmap(
    z=heatmap_data,
    x=["CFTC", "EMIR"],
    y=trade_ids,
    annotation_text=annotations,
    colorscale=[
        [0, "#f87171"],
        [1, "#4ade80"]
    ],
    showscale=False
)


fig.update_xaxes(side="bottom")

fig.update_layout(
    title="Portfolio Compliance Heatmap",
    height=600,
    font=dict(size=12)
)


fig.show()

1. Portfolio compliance heatmap


The portfolio-level compliance heatmap reveals that the vast majority of trades fail to meet regulatory requirements under both CFTC and EMIR frameworks. Only two trades, T017 and T021, achieve a compliant status under CFTC rules, while every single instrument in the portfolio is marked as non-compliant under EMIR.

In [20]:
# ======================
# 4. Chart 2：Error frequency chart
print("2. Error frequency chart")

errors = []
for c in compliance_results:
    for f in c.get("findings", []):
        errors.append(f.get("field", "unknown"))

if errors:
    df_err = pd.Series(errors).value_counts().reset_index()
    df_err.columns = ["field", "count"]
    fig2 = px.bar(df_err, x="count", y="field", orientation="h",
                   title="Error Field Frequency",
                   color="count", color_continuous_scale="Reds")
    fig2.show()
else:
    print("no error")

2. Error frequency chart


The chart shows that missing counterparty LEI fields are the most frequent compliance issues, with other_counterparty_lei having the highest error count. Errors related to EMIR-specific collateral and margin fields are the next most common. In contrast, issues with notional currency and asset class are the least frequent problems in the portfolio.

In [21]:
# ======================
# 4. Chart 3: Asset class breakdown
print("3. Asset class breakdown")

cls = summary.get("classification_counts", {})

labels = []
values = []
for key, val in cls.items():
    if key == "CONVENTIONAL_DERIVATIVE":
        labels.append("Conventional Derivatives")
    elif key == "NOVEL_INSTRUMENT_NO_TAXONOMY":
        labels.append("Novel / Unclassified Instruments")
    else:
        labels.append(key)
    values.append(val)


colors = ["#6366f1", "#f87171"]

fig3 = px.pie(
    names=labels,
    values=values,
    title="Asset Class Breakdown",
    color_discrete_sequence=colors
)


fig3.update_traces(
    textinfo="percent+label",
    textposition="outside",
    textfont={"size": 14},
    insidetextorientation="horizontal"
)

fig3.show()

3. Asset class breakdown


The classification breakdown shows that 3 out of 28 trades (10.7%) are “Novel / Unclassified Instruments”. These trades, T026, T027, and T028, are precisely the ones flagged with NO_PRODUCT_DEFINITION in the UPI taxonomy, creating fundamental ambiguity for regulators. This small but problematic subset is a major source of risk.

In [28]:
# ======================
# 5. Chart 4: Classification frontier panel

import pandas as pd
from IPython.display import display, HTML

print("4. Classification frontier panel")

table_data = []
for trade_id in novel_trade_ids:
    cftc_status = "N/A"
    emir_status = "N/A"
    cftc_notes = []
    emir_notes = []

    for c in compliance_results:
        if c["trade_id"] == trade_id:
            if c["regime"] == "CFTC":
                cftc_status = c["status"]
                for f in c.get("findings", []):
                    cftc_notes.append(f["message"])
            elif c["regime"] == "EMIR":
                emir_status = c["status"]
                for f in c.get("findings", []):
                    emir_notes.append(f["message"])

    table_data.append({
        "Trade ID": trade_id,
        "CFTC Status": cftc_status,
        "CFTC Compliance Notes": "\n".join(cftc_notes) if cftc_notes else "No issues",
        "EMIR Status": emir_status,
        "EMIR Compliance Notes": "\n".join(emir_notes) if emir_notes else "No issues"
    })

df_table = pd.DataFrame(table_data)

def highlight_status(val):
    if val == "PASS":
        return "background-color: #4ade80; color: white;"
    elif val == "CONDITIONAL":
        return "background-color: #fbbf24; color: #1f2937;"
    elif val == "NOT_APPLICABLE":
        return "background-color: #9ca3af; color: white;"
    else:
        return "background-color: #f87171; color: white;"

styled_table = df_table.style.map(
    highlight_status, subset=["CFTC Status", "EMIR Status"]
).hide(axis='index') \
.set_properties(**{
    'white-space': 'pre-wrap',
    'border': '1px solid #374151',
    'padding': '10px',
    'color': '#e5e7eb',
    'text-align': 'left'
}).set_table_styles([{
    'selector': 'th',
    'props': [
        ('background-color', '#1f2937'),
        ('color', '#f9fafb'),
        ('text-align', 'left'),
        ('padding', '12px')
    ]
}]).set_caption("T026-T028 Jurisdictional Asymmetry Compliance Table")

display(styled_table)

4. Classification frontier panel


Trade ID,CFTC Status,CFTC Compliance Notes,EMIR Status,EMIR Compliance Notes
T026,NONCOMPLIANT,other_counterparty_lei is required for CFTC. other_counterparty_lei is missing. Instrument type 'BinaryEventContract' under asset class 'EventContract' has no product definition in the ANNA-DSB OTC UPI taxonomy. Kalshi event contract is treated as conditional CFTC scope for the written analysis.,NONCOMPLIANT,other_counterparty_lei is required for EMIR. collateral_portfolio_code is required for EMIR. initial_margin_posted is required for EMIR. variation_margin_posted is required for EMIR. other_counterparty_lei is missing. Instrument type 'BinaryEventContract' under asset class 'EventContract' has no product definition in the ANNA-DSB OTC UPI taxonomy. EventContract is outside EMIR OTC derivative taxonomy in this project.
T027,NONCOMPLIANT,uti is required for CFTC. other_counterparty_lei is required for CFTC. other_counterparty_lei is missing. UTI is missing. USDC is not a valid ISO 4217 currency code. Instrument type 'BinaryEventContract' under asset class 'EventContract' has no product definition in the ANNA-DSB OTC UPI taxonomy. Non-DCM event contract is treated as not applicable for CFTC OTC reporting.,NONCOMPLIANT,uti is required for EMIR. other_counterparty_lei is required for EMIR. collateral_portfolio_code is required for EMIR. initial_margin_posted is required for EMIR. variation_margin_posted is required for EMIR. other_counterparty_lei is missing. UTI is missing. USDC is not a valid ISO 4217 currency code. Instrument type 'BinaryEventContract' under asset class 'EventContract' has no product definition in the ANNA-DSB OTC UPI taxonomy. EventContract is outside EMIR OTC derivative taxonomy in this project.
T028,NONCOMPLIANT,other_counterparty_lei is required for CFTC. reporting_counterparty_lei=9695009AXSRNHZE85Y20 is not a valid LEI: The number's checksum or check digit is invalid. other_counterparty_lei is missing. Instrument type 'BinaryEventContract' under asset class 'EventContract' has no product definition in the ANNA-DSB OTC UPI taxonomy. Kalshi event contract is treated as conditional CFTC scope for the written analysis.,NONCOMPLIANT,other_counterparty_lei is required for EMIR. collateral_portfolio_code is required for EMIR. initial_margin_posted is required for EMIR. variation_margin_posted is required for EMIR. reporting_counterparty_lei=9695009AXSRNHZE85Y20 is not a valid LEI: The number's checksum or check digit is invalid. other_counterparty_lei is missing. Instrument type 'BinaryEventContract' under asset class 'EventContract' has no product definition in the ANNA-DSB OTC UPI taxonomy. EventContract is outside EMIR OTC derivative taxonomy in this project.


In [23]:
print("All charts are illustrated.")

All charts are illustrated.
